# 4. Training, Evaluasi, & Kuantisasi INT8 MobileNetV2 DOC Sexing (Day-Old Chick)
Target Device: **Seeed Grove Vision AI Module V2 (Cortex-M55 + Ethos-U55 NPU)**

Notebook ini merupakan pipeline end-to-end khusus untuk **klasifikasi jenis kelamin anak ayam umur sehari (*Day-Old Chick* / DOC)**:
1. **Auto-Augmentasi Data**: Memperbanyak dataset mentah (16 Female & 15 Male) menjadi **1.000 citra per kelas** (disimpan di `data/doc_augmented/`).
2. **Transfer Learning**: Melatih arsitektur **MobileNetV2 ($lpha=0.35$, $160	imes160	imes3$)** dengan pemetaan `0 = Female` dan `1 = Male`.
3. **Kuantisasi Penuh INT8**: Mengekspor model terkuantisasi penuh (`model_doc_int8.tflite`) yang 100% kompatibel dengan operator Ethos-U55 NPU.
4. **Kompilasi Arm Vela**: Mengompilasi model TFLite menjadi `model_doc_int8_vela.tflite` untuk dieksekusi di NPU hardware.

> **Catatan Server / Cloud**: Seluruh path menggunakan **path relatif** terhadap lokasi repositori, sehingga dapat dijalankan di server Linux, Colab, maupun lokal tanpa modifikasi path.


## Langkah 0: Instalasi & Import Library
Jalankan sel di bawah ini jika library belum terpasang di environment server Anda.


In [ ]:
# Uncomment baris di bawah ini jika dijalankan di Google Colab / server baru
# !pip install -q tensorflow scikit-learn matplotlib seaborn pillow ethos-u-vela


In [ ]:
import os
import sys
import pathlib
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Kunci seed deterministik untuk reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Versi TensorFlow :", tf.__version__)
print("GPU Tersedia     :", tf.config.list_physical_devices('GPU'))


## Langkah 1: Konfigurasi Parameter, Path Relatif, & Auto-Augmentasi Dataset DOC
Sistem akan memeriksa folder `data/doc_augmented/`. Jika belum ada, sistem akan otomatis memicu generator augmentasi dari folder mentah `data/doc/`.


In [ ]:
# Path Relatif Portabel
NOTEBOOK_DIR = pathlib.Path.cwd()
BASE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebook" else NOTEBOOK_DIR

DATA_RAW_DIR = BASE_DIR / "data" / "doc"
DATA_AUG_DIR = BASE_DIR / "data" / "doc_augmented"
MODEL_OUTPUT_DIR = BASE_DIR / "model"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Konfigurasi Model NPU-Friendly (MobileNetV2 0.35 - 160x160)
IMG_SIZE = 160
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)
ALPHA = 0.35
BATCH_SIZE = 32

# Urutan kelas WAJIB: index 0 = 'Female' (Betina), index 1 = 'Male' (Jantan)
CLASS_NAMES = ["Female", "Male"]
CLASS_LABELS_ID = {0: "Class 0: Betina (Female)", 1: "Class 1: Jantan (Male)"}

# Epochs untuk Fase 1 (Head Training) & Fase 2 (Fine-Tuning)
EPOCHS_PHASE_1 = 10
EPOCHS_PHASE_2 = 15

print(f"Base Directory      : {BASE_DIR.resolve()}")
print(f"Dataset Augmented   : {DATA_AUG_DIR.resolve()}")
print(f"Output Model        : {MODEL_OUTPUT_DIR.resolve()}")
print(f"Pemetaan Kelas      : 0 -> {CLASS_NAMES[0]} (Betina), 1 -> {CLASS_NAMES[1]} (Jantan)")


In [ ]:
# Periksa apakah dataset augmented sudah siap (minimal 500 gambar per kelas di folder train)
train_female_count = len(list((DATA_AUG_DIR / "train" / "Female").glob("*.jpg"))) if (DATA_AUG_DIR / "train" / "Female").exists() else 0

if train_female_count < 500:
    print("[INFO] Dataset augmented belum ditemukan atau belum lengkap.")
    print("       Menjalankan modul augmentasi otomatis (target 1.000 citra/kelas)...")
    
    # Import modul augment_doc dari data/augment_doc.py
    sys.path.append(str(BASE_DIR / "data"))
    try:
        from augment_doc import generate_augmented_dataset
        generate_augmented_dataset(
            src_dir=DATA_RAW_DIR,
            out_dir=DATA_AUG_DIR,
            target_per_class=1000,
            train_ratio=0.70,
            valid_ratio=0.15,
            seed=SEED
        )
    except ImportError:
        print("[WARN] Modul augment_doc.py tidak dapat diimpor langsung, memanggil via subprocess...")
        import subprocess
        subprocess.run([
            sys.executable,
            str(BASE_DIR / "data" / "augment_doc.py"),
            "--src-dir", str(DATA_RAW_DIR),
            "--out-dir", str(DATA_AUG_DIR),
            "--target-per-class", "1000"
        ], check=True)
else:
    print(f"[OK] Dataset augmented sudah tersedia di disk ({train_female_count} sampel train Female terdeteksi).")


In [ ]:
# Memuat Dataset dengan mengunci urutan class_names=['Female', 'Male']
print("Memuat dataset train / valid / test dari:", DATA_AUG_DIR)

raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_AUG_DIR / "train",
    class_names=CLASS_NAMES,
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_AUG_DIR / "valid",
    class_names=CLASS_NAMES,
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

raw_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_AUG_DIR / "test",
    class_names=CLASS_NAMES,
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("\nVerifikasi class_names:", raw_train_ds.class_names)


## Langkah 2: Preprocessing & Visualisasi Sampel Dataset
Normalisasi input ke rentang $[-1.0, 1.0]$ sesuai format baku `MobileNetV2` sebelum masuk ke layer konvolusi.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Augmentasi ringan on-the-fly pada batch training (flip horizontal & rotasi minor)
def augment_and_preprocess(images, labels):
    images = tf.cast(images, tf.float32)
    images = tf.image.random_flip_left_right(images)
    images = (images / 127.5) - 1.0  # Normalisasi ke [-1.0, 1.0]
    return images, labels

def preprocess_only(images, labels):
    images = tf.cast(images, tf.float32)
    images = (images / 127.5) - 1.0
    return images, labels

train_ds = raw_train_ds.map(augment_and_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = raw_val_ds.map(preprocess_only, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = raw_test_ds.map(preprocess_only, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


In [ ]:
# Visualisasi 8 sampel gambar dari dataset training DOC
plt.figure(figsize=(12, 6))
for images, labels in raw_train_ds.take(1):
    for i in range(min(8, len(images))):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_idx = int(labels[i].numpy())
        plt.title(f"{CLASS_NAMES[label_idx]} (Class {label_idx})", fontsize=11, fontweight="bold")
        plt.axis("off")
plt.tight_layout()
plt.show()


## Langkah 3: Membangun Arsitektur MobileNetV2 Ringan ($lpha=0.35$)
Model menggunakan backbone **MobileNetV2 ($lpha=0.35$)** yang sangat efisien dan 100% dipetakan ke NPU Arm Ethos-U55.


In [ ]:
def build_npu_mobilenetv2_doc(input_shape=(160, 160, 3), alpha=0.35, num_classes=2):
    inputs = tf.keras.Input(shape=input_shape, name="input_image")

    # Base model MobileNetV2 ringan dengan bobot ImageNet
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        alpha=alpha,
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False  # Bekukan backbone untuk Fase 1

    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = tf.keras.layers.Dropout(0.2, name="dropout_head")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="output_classes")(x)

    model = tf.keras.Model(inputs, outputs, name="MobileNetV2_DOC_Sexing_Alpha035")
    return model, base_model

model, base_model = build_npu_mobilenetv2_doc(IMG_SHAPE, ALPHA, num_classes=len(CLASS_NAMES))
model.summary()


## Langkah 4: Training Model (2 Fase: Head Warmup & Fine-Tuning)


In [ ]:
# --- FASE 1: Training Classification Head ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

print("=" * 60)
print(f"  FASE 1: TRAINING HEAD ({EPOCHS_PHASE_1} Epochs, lr=1e-3)")
print("=" * 60)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE_1,
)


In [ ]:
# --- FASE 2: Fine-Tuning ---
base_model.trainable = True

# Bekukan semua layer awal, hanya buka 30 layer terakhir
fine_tune_at = max(0, len(base_model.layers) - 30)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print("=" * 60)
print(f"  FASE 2: FINE-TUNING ({EPOCHS_PHASE_2} Epochs, lr=1e-4)")
print(f"  Layer dibuka: {len(base_model.layers) - fine_tune_at} layer terakhir")
print("=" * 60)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE_2,
)

# Simpan model float32
keras_model_path = MODEL_OUTPUT_DIR / "mobilenetv2_doc_float32.keras"
model.save(keras_model_path)
print(f"[OK] Model Keras Float32 tersimpan: {keras_model_path}")


## Langkah 5: Evaluasi Model (Kurva Training, Confusion Matrix, & Metrik Klasifikasi)


In [ ]:
# Gabungkan riwayat training Fase 1 dan Fase 2
acc = history_phase1.history["accuracy"] + history_phase2.history["accuracy"]
val_acc = history_phase1.history["val_accuracy"] + history_phase2.history["val_accuracy"]
loss = history_phase1.history["loss"] + history_phase2.history["loss"]
val_loss = history_phase1.history["val_loss"] + history_phase2.history["val_loss"]

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label="Training Accuracy", color="#1f77b4", linewidth=2)
plt.plot(val_acc, label="Validation Accuracy", color="#ff7f0e", linewidth=2)
plt.axvline(x=EPOCHS_PHASE_1 - 1, color="gray", linestyle="--", label="Start Fine-Tuning")
plt.title("Akurasi Model DOC MobileNetV2", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(loss, label="Training Loss", color="#1f77b4", linewidth=2)
plt.plot(val_loss, label="Validation Loss", color="#ff7f0e", linewidth=2)
plt.axvline(x=EPOCHS_PHASE_1 - 1, color="gray", linestyle="--", label="Start Fine-Tuning")
plt.title("Loss Model DOC MobileNetV2", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred_float = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred_float.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred_float = np.array(y_pred_float)

print("=" * 60)
print("  EVALUASI TEST SET (MODEL FLOAT32)")
print("=" * 60)
print(classification_report(y_true, y_pred_float, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred_float)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix - Test Set DOC (Float32)", fontsize=11, fontweight="bold")
plt.xlabel("Prediksi Model")
plt.ylabel("Label Sebenarnya")
plt.tight_layout()
plt.show()


## Langkah 6: Kuantisasi Penuh ke `model_doc_int8.tflite` (100% NPU Compatible)
Kuantisasi penuh (*Full Integer Quantization INT8*) mengonversi seluruh bobot dan aktivasi ke integer 8-bit untuk akselerasi perangkat keras NPU Arm Ethos-U55.


In [ ]:
# 1. Kunci dimensi input menjadi statis [1, 160, 160, 3] agar 100% didukung NPU di Arm Vela
@tf.function(input_signature=[tf.TensorSpec(shape=[1, IMG_SIZE, IMG_SIZE, 3], dtype=tf.float32)])
def static_inference_fn(x):
    return model(x, training=False)

concrete_func = static_inference_fn.get_concrete_function()

# 2. Generator Representative Dataset dari data validasi
def representative_dataset_gen():
    sample_count = 0
    for images, _ in raw_val_ds:
        for i in range(images.shape[0]):
            single_img = tf.expand_dims(images[i], axis=0)
            single_img = tf.cast(single_img, tf.float32)
            single_img = (single_img / 127.5) - 1.0  # Skala [-1.0, 1.0]
            yield [single_img]
            sample_count += 1
            if sample_count >= 150:
                return

# 3. Konfigurasi TFLite Converter INT8 Penuh
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

# 4. Simpan ke model_doc_int8.tflite
tflite_model_path = MODEL_OUTPUT_DIR / "model_doc_int8.tflite"
with open(tflite_model_path, "wb") as f:
    f.write(tflite_quant_model)

size_kb = len(tflite_quant_model) / 1024
print(f"[SUCCESS] Model INT8 tersimpan di: {tflite_model_path.resolve()}")
print(f"          Ukuran File INT8       : {size_kb:.2f} KB")

# Tulis labels_doc.txt
labels_file_path = MODEL_OUTPUT_DIR / "labels_doc.txt"
with open(labels_file_path, "w") as f:
    for i, name in enumerate(CLASS_NAMES):
        f.write(f"{i} {name}\n")
print(f"[SUCCESS] Label tersimpan di     : {labels_file_path.resolve()}")


## Langkah 7: Evaluasi Akurasi Model `model_doc_int8.tflite`
Memverifikasi bahwa kuantisasi INT8 tidak mengalami penurunan akurasi signifikan (*accuracy drop*).


In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(tflite_model_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

in_scale, in_zero_point = input_details["quantization"]
out_scale, out_zero_point = output_details["quantization"]

print(f"Input Quantization  : Scale={in_scale:.6f}, ZeroPoint={in_zero_point}")
print(f"Output Quantization : Scale={out_scale:.6f}, ZeroPoint={out_zero_point}")

y_pred_int8 = []
y_true_int8 = []

for images, labels in raw_test_ds:
    for i in range(images.shape[0]):
        img_raw = images[i].numpy().astype(np.float32)
        img_norm = (img_raw / 127.5) - 1.0
        img_int8 = np.clip(np.round(img_norm / in_scale + in_zero_point), -128, 127).astype(np.int8)
        img_int8 = np.expand_dims(img_int8, axis=0)

        interpreter.set_tensor(input_details["index"], img_int8)
        interpreter.invoke()
        output_int8 = interpreter.get_tensor(output_details["index"])[0]

        pred_class = int(np.argmax(output_int8))
        y_pred_int8.append(pred_class)
        y_true_int8.append(int(labels[i].numpy()))

y_true_int8 = np.array(y_true_int8)
y_pred_int8 = np.array(y_pred_int8)

int8_acc = np.mean(y_true_int8 == y_pred_int8) * 100.0
print("=" * 60)
print(f"  AKURASI MODEL INT8 PADA TEST SET : {int8_acc:.2f}%")
print("=" * 60)
print(classification_report(y_true_int8, y_pred_int8, target_names=CLASS_NAMES, digits=4))


## Langkah 8: Kompilasi dengan Arm Vela Compiler & Verifikasi 100% NPU
Mengompilasi `model_doc_int8.tflite` dengan **Arm Ethos-U55 Vela Compiler** untuk akselerasi perangkat keras on-device.


In [ ]:
import subprocess
import shutil

vela_bin = shutil.which("vela")
vela_out_dir = MODEL_OUTPUT_DIR / "vela_out"
vela_out_dir.mkdir(parents=True, exist_ok=True)

if vela_bin is not None:
    print(f"Arm Vela Compiler terdeteksi di: {vela_bin}")
    cmd = [
        vela_bin,
        str(tflite_model_path),
        "--accelerator-config", "ethos-u55-128",
        "--system-config", "Ethos_U55_High_End_Embedded",
        "--memory-mode", "Shared_Sram",
        "--output-dir", str(vela_out_dir),
    ]
    print("Menjalankan Vela:", " ".join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.returncode == 0:
        compiled_model = vela_out_dir / "model_doc_int8_vela.tflite"
        print(f"[SUCCESS] Model Vela NPU selesai: {compiled_model}")
    else:
        print("[WARN] Vela kompilasi error:\n", res.stderr)
else:
    print("[INFO] 'vela' belum terinstal di environment lokal. Jalankan di server/terminal:")
    print(f"       vela {tflite_model_path} --accelerator-config ethos-u55-128 --output-dir {vela_out_dir}")
